In [1]:
# --- Portable data location (added by macos-native/tools/mac_python_paths.py) ---------------------
# RAMAN_HANDOFF_DIR is the hand-off folder laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug,
# found by walking up from this file (a notebook: from its working folder). To use another copy,
# set the RAMAN_HANDOFF_DIR environment variable to that folder.
import os as _os


def _raman_handoff_dir():
    if _os.environ.get("RAMAN_HANDOFF_DIR"):
        return _os.path.abspath(_os.environ["RAMAN_HANDOFF_DIR"])
    start = _os.path.dirname(_os.path.abspath(globals()["__file__"])) if "__file__" in globals() else _os.getcwd()
    folder = start
    while True:
        candidate = _os.path.join(folder, "laser_test", "build", "Desktop_Qt_6_6_3_MSVC2019_64bit-Debug")
        if _os.path.isdir(candidate):
            return candidate
        parent = _os.path.dirname(folder)
        if parent == folder:
            raise FileNotFoundError("laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug not found above "
                                    + start + "; set the RAMAN_HANDOFF_DIR environment variable")
        folder = parent


RAMAN_HANDOFF_DIR = _raman_handoff_dir()
# ---------------------------------------------------------------------------------------------------
"""
Raman mixture detection with Transformer, matched to the current PCA-SVM workflow.

Goal:
  - Use the same baseline_raw data used by the PCA-SVM mixture branch.
  - Treat di_water as blank/background, not as a reportable target component.
  - Train multi-label models to detect target analytes:
        r6g, methylene_blue, adenine
  - This file runs the Transformer backbone only. Data generation, labels, thresholds, and final decision match the PCA-SVM workflow.

Recommended comparison:
  1. PCA-SVM mixture branch: baseline_raw -> StandardScaler -> PCA + peak features -> OvR SVM
  2. CNN mixture model:      baseline_raw -> spectral StandardScaler -> 1D ResNet -> multi-label sigmoid
  3. Transformer model:      baseline_raw -> spectral StandardScaler -> 1D Transformer -> multi-label sigmoid

Before running:
  Make sure these files exist in DATA_DIR:
    training_spectra_r6g_baseline_raw.npy
    training_spectra_methylene_blue_baseline_raw.npy
    training_spectra_adenine_baseline_raw.npy
    training_spectra_di_water_baseline_raw.npy
"""

import os
import sys
import re
import json
import time
import math
import random
from pathlib import Path
from itertools import combinations
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =========================================================
# CONFIG
# =========================================================
DATA_ROOT = RAMAN_HANDOFF_DIR
DATA_DIR = os.path.join(DATA_ROOT, "data_base_code_part")

# Run only Transformer in this file
MODEL_TYPE = "transformer"
RETRAIN_MODEL = True

# Baseline_raw training files.
# These are Raw - Dark -> Drop -> Baseline correction, without per-spectrum min-max.
TRAINING_RAW_NPY_FILES = {
    "r6g": os.path.join(DATA_DIR, "training_spectra_r6g_baseline_raw.npy"),
    "methylene_blue": os.path.join(DATA_DIR, "training_spectra_methylene_blue_baseline_raw.npy"),
    "adenine": os.path.join(DATA_DIR, "training_spectra_adenine_baseline_raw.npy"),
    "di_water": os.path.join(DATA_DIR, "training_spectra_di_water_baseline_raw.npy"),
}

# Final report only includes these target analytes.
# di_water is used as blank/background control and negative sample.
TARGET_ANALYTES = ["r6g", "methylene_blue", "adenine"]
BACKGROUND_CLASSES = ["di_water"]

# Test file and dark file.
TEST_FILE = f"{RAMAN_HANDOFF_DIR}/portable_raman_test_data/laser_log_20260513_150549.txt"
TEST_DARK_FILE = f"{RAMAN_HANDOFF_DIR}/portable_raman_test_data/dark_20260513_150739.txt"

# Must match your current preprocessing code.
TARGET_LEN_RAW = 2048
DROP_N = 90
DO_BASELINE = True
BASELINE_WINDOW = 15
BASELINE_SMOOTH = 15

# Split and training.
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20
BATCH_SIZE = 32
EPOCHS_CNN = 80
EPOCHS_TRANSFORMER = 40
LR_CNN = 5e-4
LR_TRANSFORMER = 1e-4
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 0
EARLY_STOP_PATIENCE = 16
GRAD_CLIP_NORM = 3.0
LABEL_SMOOTHING_BCE = 0.02

# Synthetic mixture generation from baseline_raw.
N_SYNTHETIC_PER_COMBO = 200
N_SYNTHETIC_TERNARY = 100
MIXTURE_RATIO_MIN = 0.10
MIXTURE_RATIO_MAX = 0.90
SYNTH_NOISE_STD_RATIO = 0.01

# Extra weak-r6g cases inside methylene_blue.
ENABLE_EXTRA_LOW_R6G_MB = True
N_EXTRA_LOW_R6G_MB = 300
LOW_R6G_RATIOS = [0.01, 0.02, 0.05, 0.10, 0.20]

# Runtime quality filter.
ENABLE_QUALITY_FILTER = True
RAW_MIN_PTP = 100.0
RAW_MIN_STD = 15.0

# Final decision thresholds.
OVR_FINAL_THRESHOLD = 0.40
R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED = 0.25

# Data augmentation on baseline_raw spectra.
USE_AUGMENT = True
AUG_SHIFT_MAX = 6
AUG_SCALE_LOW = 0.95
AUG_SCALE_HIGH = 1.05
AUG_NOISE_STD_RATIO = 0.01
AUG_BASELINE_DRIFT_RATIO = 0.02

# CNN backbone.
CNN_HIDDEN_CHANNELS = 96
CNN_EMBED_DIM = 128

# Transformer backbone.
DOWNSAMPLE_STRIDE = 32
TRANSFORMER_CHANNELS = 48
TRANSFORMER_HEADS = 4
TRANSFORMER_FF_DIM = 96
TRANSFORMER_BLOCKS = 2
TRANSFORMER_DROPOUT = 0.20
TRANSFORMER_MLP_DROPOUT = 0.30
TRANSFORMER_EMBED_DIM = 128

OUTPUT_DIR = os.path.join(DATA_ROOT, "_Raman_multilabel_CNN_Transformer_target_only")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE.startswith("cuda")

print("DEVICE =", DEVICE)
print("MODEL_TYPE =", MODEL_TYPE)
print("OUTPUT_DIR =", OUTPUT_DIR)


# =========================================================
# UTILS
# =========================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def _is_jupyter() -> bool:
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell in ("ZMQInteractiveShell", "TerminalInteractiveShell"):
            return True
    except Exception:
        pass
    return len(sys.argv) > 0 and "jupyter" in sys.argv[0].lower()


def _parse_cli_args() -> Tuple[str, str]:
    if _is_jupyter():
        return TEST_FILE, TEST_DARK_FILE
    args = [a for a in sys.argv[1:] if not a.startswith("-")]
    tf = args[0].strip() if len(args) >= 1 and args[0].strip() else TEST_FILE
    td = args[1].strip() if len(args) >= 2 and args[1].strip() else TEST_DARK_FILE
    return tf, td


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -50, 50)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


# =========================================================
# PORTABLE RAMAN PARSING AND PREPROCESSING
# =========================================================
ERROR_PHRASE = "image data error code 0x0001"


def strip_prefix(line: str) -> str:
    line = line.strip()
    line = re.sub(r"^\[[^\]]+\]\s+\[Backend\]\s*", "", line)
    line = re.sub(r"^\[[^\]]+\]\s*", "", line)
    return line.strip()


def load_dark_spectrum_from_file(dark_path: str) -> np.ndarray:
    if not dark_path or not Path(dark_path).exists():
        return np.array([], dtype=np.float64)

    intensities: List[float] = []
    with open(dark_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("index"):
                continue
            parts = line.split("\t")
            if len(parts) >= 3:
                try:
                    intensities.append(float(parts[2]))
                except ValueError:
                    continue

    return np.asarray(intensities, dtype=np.float64)


def parse_portable_log_to_spectra(log_file: str) -> Tuple[List[np.ndarray], int]:
    lines = open(log_file, "r", encoding="utf-8", errors="ignore").read().splitlines()
    spectra: List[np.ndarray] = []
    failed = 0
    i = 0

    while i < len(lines):
        line_raw = lines[i]
        line = strip_prefix(line_raw)

        if ERROR_PHRASE in line_raw.lower():
            failed += 1
            i += 1
            continue

        if line.startswith("Image Data"):
            tokens = list(re.findall(r"\b[0-9A-Fa-f]{2,4}\b", line[len("Image Data"):].strip()))
            j = i + 1
            block_err = False

            while j < len(lines):
                nxt_raw = lines[j]
                nxt = strip_prefix(nxt_raw)

                if ERROR_PHRASE in nxt_raw.lower():
                    block_err = True

                if nxt.startswith("Stop Capture") or (nxt.startswith("Image Data") and j != i):
                    break

                hexes = re.findall(r"\b[0-9A-Fa-f]{2,4}\b", nxt)
                if hexes:
                    tokens.extend(hexes)

                j += 1

            if block_err or not tokens:
                failed += 1
            else:
                vals = np.array([int(t, 16) for t in tokens], dtype=np.int64)

                if vals.size % 2 == 0 and vals.max(initial=0) <= 255:
                    le = vals[0::2] + (vals[1::2] << 8)
                    be = (vals[0::2] << 8) + vals[1::2]
                    v16 = le if le.max(initial=0) >= be.max(initial=0) else be
                    if v16.max(initial=0) > 255:
                        vals = v16.astype(np.int64)

                spectra.append(vals.astype(np.float32))

            i = j + 1
            continue

        i += 1

    return spectra, failed


def moving_average(y: np.ndarray, window: int) -> np.ndarray:
    window = int(max(3, window))
    if window % 2 == 0:
        window += 1
    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / window
    return np.convolve(ypad, kernel, mode="valid")


def rolling_minimum(y: np.ndarray, window: int) -> np.ndarray:
    window = int(max(3, window))
    if window % 2 == 0:
        window += 1
    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    out = np.empty_like(y, dtype=np.float64)
    for i in range(y.size):
        out[i] = np.min(ypad[i:i + window])
    return out


def baseline_correction_fast(y: np.ndarray, window: int = 15, smooth: int = 15) -> np.ndarray:
    y = np.asarray(y, dtype=np.float64)
    if y.size < 5:
        return y.copy().astype(np.float32)
    baseline = rolling_minimum(y, window=window)
    baseline = moving_average(baseline, window=smooth)
    return (y - baseline).astype(np.float32)


def preprocess_baseline_raw(raw_spectrum: np.ndarray, dark_raw: np.ndarray) -> Tuple[Optional[np.ndarray], Optional[Dict[str, float]]]:
    s = np.asarray(raw_spectrum, dtype=np.float64)
    if s.size < TARGET_LEN_RAW:
        return None, None

    s2048 = s[:TARGET_LEN_RAW]
    if dark_raw.size >= TARGET_LEN_RAW:
        raw_dark_full = s2048 - dark_raw[:TARGET_LEN_RAW]
    else:
        raw_dark_full = s2048.copy()

    if DROP_N >= raw_dark_full.size:
        return None, None

    raw_dark = raw_dark_full[DROP_N:]
    quality = {
        "ptp": float(np.ptp(raw_dark)),
        "std": float(np.std(raw_dark)),
    }

    if DO_BASELINE:
        baseline_raw = baseline_correction_fast(raw_dark, window=BASELINE_WINDOW, smooth=BASELINE_SMOOTH)
    else:
        baseline_raw = raw_dark.astype(np.float32)

    return baseline_raw.astype(np.float32), quality


# =========================================================
# DATA LOADING AND SYNTHETIC MIXTURE GENERATION
# =========================================================
all_class_names = list(TRAINING_RAW_NPY_FILES.keys())
all_class_to_id = {c: i for i, c in enumerate(all_class_names)}
all_id_to_class = {i: c for c, i in all_class_to_id.items()}

for name in TARGET_ANALYTES:
    if name not in all_class_to_id:
        raise ValueError(f"TARGET_ANALYTES contains {name}, but it is not in TRAINING_RAW_NPY_FILES")

target_names = TARGET_ANALYTES.copy()
target_to_label = {c: i for i, c in enumerate(target_names)}
label_to_target = {i: c for c, i in target_to_label.items()}
NUM_TARGETS = len(target_names)


def make_multilabel_for_classes(class_list: List[str]) -> np.ndarray:
    y = np.zeros(NUM_TARGETS, dtype=np.float32)
    for cls in class_list:
        if cls in target_to_label:
            y[target_to_label[cls]] = 1.0
    return y


def load_raw_training_data() -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    print("\n" + "=" * 70)
    print("Step 1: Load baseline_raw pure data")
    print("=" * 70)

    data_by_class: Dict[str, np.ndarray] = {}
    X_parts: List[np.ndarray] = []
    y_single_parts: List[np.ndarray] = []

    for cls in all_class_names:
        path = TRAINING_RAW_NPY_FILES[cls]
        if not os.path.exists(path):
            raise FileNotFoundError(f"Cannot find {cls} baseline_raw data: {path}")
        data = np.load(path).astype(np.float32)
        data_by_class[cls] = data
        X_parts.append(data)
        y_single_parts.append(np.full(data.shape[0], all_class_to_id[cls], dtype=np.int64))
        print(f"  {cls:20s} shape={data.shape}")

    X_pure_raw = np.vstack(X_parts).astype(np.float32)
    Y_pure_single = np.concatenate(y_single_parts)

    print(f"\n  Pure raw total: {X_pure_raw.shape}")
    print(f"  Target analytes: {target_names}")
    print(f"  Background classes: {BACKGROUND_CLASSES}")

    expected_len = TARGET_LEN_RAW - DROP_N
    if X_pure_raw.shape[1] != expected_len:
        print(f"  Warning: feature length is {X_pure_raw.shape[1]}, expected {expected_len} from 2048-DROP_N.")
        print("  This is OK only if your data generation used a different DROP_N or crop length.")

    return data_by_class, X_pure_raw, Y_pure_single


def generate_synthetic_mixture_dataset(data_by_class: Dict[str, np.ndarray], X_pure_raw: np.ndarray, Y_pure_single: np.ndarray):
    print("\n" + "=" * 70)
    print("Step 2: Generate synthetic mixtures from baseline_raw")
    print("=" * 70)

    rng = np.random.RandomState(SEED)
    X_list: List[np.ndarray] = []
    Y_list: List[np.ndarray] = []

    # Add pure spectra.
    for i in range(X_pure_raw.shape[0]):
        cls_name = all_id_to_class[int(Y_pure_single[i])]
        X_list.append(X_pure_raw[i].reshape(1, -1))
        Y_list.append(make_multilabel_for_classes([cls_name]))

    n_pure = X_pure_raw.shape[0]
    n_binary = 0
    n_ternary = 0

    # Binary mixtures among all classes, including di_water.
    # di_water contributes background shape, but not a reportable label.
    binary_combos = list(combinations(all_class_names, 2))
    print(f"\n  Binary combinations: {len(binary_combos)}")

    for cls_a, cls_b in binary_combos:
        A = data_by_class[cls_a]
        B = data_by_class[cls_b]
        print(f"    {cls_a} + {cls_b}: {N_SYNTHETIC_PER_COMBO}")

        for _ in range(N_SYNTHETIC_PER_COMBO):
            spec_a = A[rng.randint(0, len(A))]
            spec_b = B[rng.randint(0, len(B))]
            ratio_a = rng.uniform(MIXTURE_RATIO_MIN, MIXTURE_RATIO_MAX)
            ratio_b = 1.0 - ratio_a
            mix = ratio_a * spec_a + ratio_b * spec_b

            if SYNTH_NOISE_STD_RATIO > 0:
                noise_std = SYNTH_NOISE_STD_RATIO * max(float(np.std(mix)), 1e-6)
                mix = mix + rng.normal(0, noise_std, size=mix.shape).astype(np.float32)

            X_list.append(mix.reshape(1, -1))
            Y_list.append(make_multilabel_for_classes([cls_a, cls_b]))
            n_binary += 1

    # Extra weak r6g in methylene_blue.
    if ENABLE_EXTRA_LOW_R6G_MB and "r6g" in data_by_class and "methylene_blue" in data_by_class:
        print("\n  Extra low-ratio r6g + methylene_blue mixtures:")
        A = data_by_class["r6g"]
        B = data_by_class["methylene_blue"]
        per_ratio = max(1, N_EXTRA_LOW_R6G_MB // len(LOW_R6G_RATIOS))

        for r_r6g in LOW_R6G_RATIOS:
            print(f"    r6g={r_r6g:.2f}, methylene_blue={1-r_r6g:.2f}: {per_ratio}")
            for _ in range(per_ratio):
                spec_r6g = A[rng.randint(0, len(A))]
                spec_mb = B[rng.randint(0, len(B))]
                mix = r_r6g * spec_r6g + (1.0 - r_r6g) * spec_mb
                if SYNTH_NOISE_STD_RATIO > 0:
                    noise_std = SYNTH_NOISE_STD_RATIO * max(float(np.std(mix)), 1e-6)
                    mix = mix + rng.normal(0, noise_std, size=mix.shape).astype(np.float32)
                X_list.append(mix.reshape(1, -1))
                Y_list.append(make_multilabel_for_classes(["r6g", "methylene_blue"]))
                n_binary += 1

    # Ternary mixtures.
    if N_SYNTHETIC_TERNARY > 0:
        ternary_combos = list(combinations(all_class_names, 3))
        print(f"\n  Ternary combinations: {len(ternary_combos)}")

        for combo in ternary_combos:
            print(f"    {' + '.join(combo)}: {N_SYNTHETIC_TERNARY}")
            arrays = [data_by_class[c] for c in combo]
            for _ in range(N_SYNTHETIC_TERNARY):
                specs = [arr[rng.randint(0, len(arr))] for arr in arrays]
                while True:
                    ratios = rng.dirichlet([1.0] * len(combo))
                    if all(r >= MIXTURE_RATIO_MIN for r in ratios):
                        break
                mix = sum(r * s for r, s in zip(ratios, specs))
                if SYNTH_NOISE_STD_RATIO > 0:
                    noise_std = SYNTH_NOISE_STD_RATIO * max(float(np.std(mix)), 1e-6)
                    mix = mix + rng.normal(0, noise_std, size=mix.shape).astype(np.float32)
                X_list.append(mix.reshape(1, -1))
                Y_list.append(make_multilabel_for_classes(list(combo)))
                n_ternary += 1

    X = np.vstack(X_list).astype(np.float32)
    Y = np.vstack(Y_list).astype(np.float32)

    print("\n  Summary:")
    print(f"    pure spectra:       {n_pure}")
    print(f"    binary synthetic:   {n_binary}")
    print(f"    ternary synthetic:  {n_ternary}")
    print(f"    total samples:      {X.shape[0]}")
    print(f"    X shape: {X.shape}, Y shape: {Y.shape}")
    for i, name in enumerate(target_names):
        print(f"    positive {name:16s}: {int(Y[:, i].sum())}")

    return X, Y


def random_split_indices(n: int):
    rng = random.Random(SEED)
    idx = list(range(n))
    rng.shuffle(idx)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    idx_train = idx[:n_train]
    idx_val = idx[n_train:n_train + n_val]
    idx_test = idx[n_train + n_val:]
    return idx_train, idx_val, idx_test


# =========================================================
# STANDARDIZATION AND DATASET
# =========================================================
class SpectralStandardizer:
    def __init__(self):
        self.mean_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

    def fit(self, X: np.ndarray):
        self.mean_ = X.mean(axis=0).astype(np.float32)
        self.std_ = X.std(axis=0).astype(np.float32)
        self.std_[self.std_ < 1e-6] = 1.0
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("Standardizer has not been fitted.")
        Z = (X.astype(np.float32) - self.mean_) / self.std_
        Z = np.clip(Z, -8.0, 8.0)
        return Z.astype(np.float32)

    def save(self, path: str):
        np.savez(path, mean=self.mean_, std=self.std_)

    @staticmethod
    def load(path: str):
        obj = SpectralStandardizer()
        data = np.load(path)
        obj.mean_ = data["mean"].astype(np.float32)
        obj.std_ = data["std"].astype(np.float32)
        return obj


def augment_raw_spectrum(x: np.ndarray) -> np.ndarray:
    x = x.copy().astype(np.float32)

    scale = np.float32(np.random.uniform(AUG_SCALE_LOW, AUG_SCALE_HIGH))
    x = x * scale

    shift = int(np.random.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1))
    if shift != 0:
        x = np.roll(x, shift)

    s = max(float(np.std(x)), 1e-6)
    x = x + np.random.normal(0, AUG_NOISE_STD_RATIO * s, size=x.shape).astype(np.float32)

    ptp = max(float(np.ptp(x)), 1e-6)
    drift = np.linspace(-1, 1, x.size, dtype=np.float32)
    amp = np.float32(np.random.uniform(-AUG_BASELINE_DRIFT_RATIO, AUG_BASELINE_DRIFT_RATIO) * ptp)
    x = x + amp * drift

    return x.astype(np.float32)


class RamanMultiLabelDataset(Dataset):
    def __init__(self, X_raw: np.ndarray, Y: np.ndarray, indices: List[int], standardizer: SpectralStandardizer, augment: bool):
        self.X_raw = X_raw
        self.Y = Y
        self.indices = indices
        self.standardizer = standardizer
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        x = self.X_raw[i].copy()
        if self.augment:
            x = augment_raw_spectrum(x)
        x = self.standardizer.transform(x.reshape(1, -1))[0]
        y = self.Y[i].astype(np.float32)
        return torch.from_numpy(x).unsqueeze(0), torch.from_numpy(y)


# =========================================================
# CNN MODEL
# =========================================================
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, stride=1, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class RamanResNetMultiLabel(nn.Module):
    def __init__(self, num_targets: int = NUM_TARGETS, hidden: int = CNN_HIDDEN_CHANNELS, embed_dim: int = CNN_EMBED_DIM):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.layers = nn.Sequential(
            ResidualBlock1D(64, hidden, stride=1),
            ResidualBlock1D(hidden, hidden, stride=1),
            ResidualBlock1D(hidden, hidden, stride=2),
            ResidualBlock1D(hidden, hidden, stride=1),
            ResidualBlock1D(hidden, hidden, stride=2),
            ResidualBlock1D(hidden, hidden, stride=1),
            ResidualBlock1D(hidden, hidden, stride=2),
            ResidualBlock1D(hidden, hidden, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.embed = nn.Sequential(
            nn.Linear(hidden, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.25),
        )
        self.head = nn.Linear(embed_dim, num_targets)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_embedding(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x).flatten(1)
        x = self.embed(x)
        return x

    def forward(self, x):
        emb = self.forward_embedding(x)
        logits = self.head(emb)
        return logits, emb


# =========================================================
# TRANSFORMER MODEL
# =========================================================
class TransformerEncoderBlock(nn.Module):
    def __init__(self, channels: int, num_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.mha = nn.MultiheadAttention(channels, num_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(channels, eps=1e-6)
        self.conv1 = nn.Conv1d(channels, ff_dim, kernel_size=1)
        self.drop2 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(ff_dim, channels, kernel_size=1)
        self.ln2 = nn.LayerNorm(channels, eps=1e-6)

    def forward(self, x):
        attn, _ = self.mha(x, x, x, need_weights=False)
        attn = self.drop1(attn)
        res = x + self.ln1(attn)

        y = res.transpose(1, 2)
        y = F.gelu(self.conv1(y))
        y = self.drop2(y)
        y = self.conv2(y).transpose(1, 2)
        out = res + self.ln2(y)
        return out


class RamanTransformerMultiLabel(nn.Module):
    def __init__(self, num_targets: int = NUM_TARGETS):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, TRANSFORMER_CHANNELS, kernel_size=7, stride=DOWNSAMPLE_STRIDE, padding=3, bias=False),
            nn.BatchNorm1d(TRANSFORMER_CHANNELS),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                channels=TRANSFORMER_CHANNELS,
                num_heads=TRANSFORMER_HEADS,
                ff_dim=TRANSFORMER_FF_DIM,
                dropout=TRANSFORMER_DROPOUT,
            )
            for _ in range(TRANSFORMER_BLOCKS)
        ])
        self.mlp = nn.Sequential(
            nn.Linear(TRANSFORMER_CHANNELS, TRANSFORMER_EMBED_DIM),
            nn.ReLU(),
            nn.Dropout(TRANSFORMER_MLP_DROPOUT),
        )
        self.head = nn.Linear(TRANSFORMER_EMBED_DIM, num_targets)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_embedding(self, x):
        x = self.stem(x)          # [B, C, L]
        x = x.transpose(1, 2)     # [B, L, C]
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1)
        x = self.mlp(x)
        return x

    def forward(self, x):
        emb = self.forward_embedding(x)
        logits = self.head(emb)
        return logits, emb


def build_model(model_type: str) -> nn.Module:
    model_type = model_type.lower()
    if model_type == "cnn":
        return RamanResNetMultiLabel(num_targets=NUM_TARGETS)
    if model_type == "transformer":
        return RamanTransformerMultiLabel(num_targets=NUM_TARGETS)
    raise ValueError(f"Unknown model_type: {model_type}")


# =========================================================
# TRAIN / EVALUATE
# =========================================================
def bce_loss_with_smoothing(logits: torch.Tensor, targets: torch.Tensor, pos_weight: Optional[torch.Tensor]):
    if LABEL_SMOOTHING_BCE > 0:
        targets = targets * (1.0 - LABEL_SMOOTHING_BCE) + 0.5 * LABEL_SMOOTHING_BCE
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)


@torch.no_grad()
def evaluate_multilabel(model: nn.Module, loader: DataLoader, threshold: float = OVR_FINAL_THRESHOLD):
    model.eval()
    all_probs = []
    all_y = []

    for x, y in loader:
        x = x.to(DEVICE)
        logits, _ = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_y.append(y.numpy())

    probs = np.vstack(all_probs)
    y_true = np.vstack(all_y).astype(np.int64)
    y_pred = (probs >= threshold).astype(np.int64)

    exact = float(np.mean(np.all(y_pred == y_true, axis=1)))

    per_class = {}
    for i, name in enumerate(target_names):
        tp = int(((y_pred[:, i] == 1) & (y_true[:, i] == 1)).sum())
        fp = int(((y_pred[:, i] == 1) & (y_true[:, i] == 0)).sum())
        fn = int(((y_pred[:, i] == 0) & (y_true[:, i] == 1)).sum())
        tn = int(((y_pred[:, i] == 0) & (y_true[:, i] == 0)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        per_class[name] = {
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall, "f1": f1,
        }

    macro_f1 = float(np.mean([v["f1"] for v in per_class.values()]))
    return {
        "exact_match": exact,
        "macro_f1": macro_f1,
        "per_class": per_class,
        "probs": probs,
        "y_true": y_true,
        "y_pred": y_pred,
    }


def train_one_model(model_type: str, X: np.ndarray, Y: np.ndarray, idx_train: List[int], idx_val: List[int], idx_test: List[int]):
    print("\n" + "=" * 70)
    print(f"Train {model_type.upper()} multi-label target detector")
    print("=" * 70)

    model_dir = os.path.join(OUTPUT_DIR, model_type.lower())
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f"{model_type.lower()}_multilabel.pt")
    scaler_path = os.path.join(model_dir, "spectral_standardizer.npz")
    meta_path = os.path.join(model_dir, "meta.json")

    standardizer = SpectralStandardizer().fit(X[idx_train])
    standardizer.save(scaler_path)

    train_loader = DataLoader(
        RamanMultiLabelDataset(X, Y, idx_train, standardizer, augment=USE_AUGMENT),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        RamanMultiLabelDataset(X, Y, idx_val, standardizer, augment=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    test_loader = DataLoader(
        RamanMultiLabelDataset(X, Y, idx_test, standardizer, augment=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    model = build_model(model_type).to(DEVICE)

    if (not RETRAIN_MODEL) and os.path.exists(model_path):
        print(f"Loading existing model: {model_path}")
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.eval()
    else:
        pos = Y[idx_train].sum(axis=0)
        neg = len(idx_train) - pos
        pos_weight_np = neg / np.maximum(pos, 1.0)
        pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)
        print("  pos_weight:", {target_names[i]: float(pos_weight_np[i]) for i in range(NUM_TARGETS)})

        lr = LR_CNN if model_type.lower() == "cnn" else LR_TRANSFORMER
        epochs = EPOCHS_CNN if model_type.lower() == "cnn" else EPOCHS_TRANSFORMER

        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5, min_lr=1e-6)

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  trainable params: {n_params:,}")

        best_score = -1.0
        best_state = None
        bad = 0

        for ep in range(1, epochs + 1):
            model.train()
            losses = []
            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                logits, _ = model(xb)
                loss = bce_loss_with_smoothing(logits, yb, pos_weight)
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()
                losses.append(float(loss.item()))

            val_metrics = evaluate_multilabel(model, val_loader)
            score = val_metrics["macro_f1"]
            scheduler.step(score)
            print(
                f"  epoch {ep:03d}/{epochs} "
                f"loss={np.mean(losses):.4f} "
                f"val_exact={val_metrics['exact_match']:.3f} "
                f"val_macro_f1={val_metrics['macro_f1']:.3f} "
                f"lr={opt.param_groups[0]['lr']:.2e}"
            )

            if score > best_score + 1e-6:
                best_score = score
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= EARLY_STOP_PATIENCE:
                    print("  early stop")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        torch.save(model.state_dict(), model_path)
        print(f"  saved model: {model_path}")

    test_metrics = evaluate_multilabel(model, test_loader)
    print("\n  TEST metrics:")
    print(f"    exact_match = {test_metrics['exact_match']:.4f}")
    print(f"    macro_f1    = {test_metrics['macro_f1']:.4f}")
    for name, m in test_metrics["per_class"].items():
        print(
            f"    {name:16s} P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} "
            f"TP={m['tp']} FP={m['fp']} FN={m['fn']} TN={m['tn']}"
        )

    meta = {
        "model_type": model_type,
        "target_analytes": target_names,
        "background_classes": BACKGROUND_CLASSES,
        "input": "baseline_raw + training-set spectral standardization",
        "feature_dim": int(X.shape[1]),
        "threshold": OVR_FINAL_THRESHOLD,
        "r6g_weak_threshold_when_mb_detected": R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED,
        "test_exact_match": float(test_metrics["exact_match"]),
        "test_macro_f1": float(test_metrics["macro_f1"]),
        "per_class": test_metrics["per_class"],
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return model, standardizer, model_dir, test_metrics


# =========================================================
# REAL FILE INFERENCE
# =========================================================
@torch.no_grad()
def predict_real_file(model: nn.Module, standardizer: SpectralStandardizer, model_type: str, spectrum_path: str, dark_path: str, debug: bool = True):
    print("\n" + "=" * 70)
    print(f"Inference on real test file with {model_type.upper()}")
    print("=" * 70)
    print(f"  Test file: {spectrum_path}")
    print(f"  Dark file: {dark_path}")

    if not os.path.exists(spectrum_path):
        raise FileNotFoundError(spectrum_path)

    dark_raw = load_dark_spectrum_from_file(dark_path) if dark_path else np.array([])
    spectra, failed = parse_portable_log_to_spectra(spectrum_path)
    print(f"\n  Parsed: {len(spectra)} spectra, {failed} failed")

    model.eval()
    per_block = []
    rejected = []

    for idx, raw in enumerate(spectra):
        v_raw, quality = preprocess_baseline_raw(raw, dark_raw)

        if v_raw is None or np.allclose(v_raw, 0):
            rejected.append({"idx": idx, "reason": "preprocessing failed"})
            continue

        if ENABLE_QUALITY_FILTER:
            if quality["ptp"] < RAW_MIN_PTP or quality["std"] < RAW_MIN_STD:
                rejected.append({"idx": idx, "reason": f"low signal, ptp={quality['ptp']:.1f}, std={quality['std']:.1f}"})
                continue

        x = standardizer.transform(v_raw.reshape(1, -1))[0]
        xt = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits, _ = model(xt)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()

        per_block.append({
            "idx": idx,
            "probs": probs,
            "quality": quality,
        })

        if debug:
            prob_str = "  ".join(f"{target_names[i]}={probs[i]:.3f}" for i in range(NUM_TARGETS))
            print(f"  [SPEC {idx:03d}] {prob_str}")

    print("\n" + "=" * 70)
    print(f"FINAL DECISION, {model_type.upper()}")
    print("=" * 70)
    print(f"  Total spectra: {len(spectra)}, parse-failed: {failed}")
    print(f"  Accepted: {len(per_block)} | Rejected: {len(rejected)}")

    if rejected:
        reason_counts: Dict[str, int] = {}
        for r in rejected:
            key = r["reason"].split(",")[0].strip()
            reason_counts[key] = reason_counts.get(key, 0) + 1
        print("  Rejection reasons:")
        for k, v in reason_counts.items():
            print(f"    {k}: {v}")

    if not per_block:
        print("\n  >>> RESULT: UNKNOWN")
        return None

    avg_probs = np.mean(np.stack([b["probs"] for b in per_block]), axis=0)

    print("\n  Average target probabilities:")
    for i, name in enumerate(target_names):
        mark = " ***" if avg_probs[i] >= OVR_FINAL_THRESHOLD else ""
        print(f"    {name:20s} {avg_probs[i]:.4f}{mark}")

    detected = set()
    for i, name in enumerate(target_names):
        if avg_probs[i] >= OVR_FINAL_THRESHOLD:
            detected.add(name)

    if "methylene_blue" in detected and avg_probs[target_to_label["r6g"]] >= R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED:
        detected.add("r6g")

    if not detected:
        print("\n  >>> RESULT: UNKNOWN")
        print("  >>> No target analyte crossed the decision threshold.")
    elif len(detected) == 1:
        name = list(detected)[0]
        print(f"\n  >>> RESULT: {name}")
        print(f"  >>> Probability: {avg_probs[target_to_label[name]]:.4f}")
    else:
        ordered = sorted(list(detected), key=lambda n: avg_probs[target_to_label[n]], reverse=True)
        print(f"\n  >>> RESULT: MIXTURE -> {' + '.join(ordered)}")
        print("  >>> Detected target components:")
        for name in ordered:
            p = avg_probs[target_to_label[name]]
            note = " weak-positive" if name == "r6g" and p < OVR_FINAL_THRESHOLD else ""
            print(f"      {name:20s} prob={p:.4f}{note}")

    # Plot per-block probabilities.
    xs = [b["idx"] for b in per_block]
    plt.figure(figsize=(12, 5))
    for i, name in enumerate(target_names):
        ys = [b["probs"][i] for b in per_block]
        plt.plot(xs, ys, marker="o", markersize=4, label=name, alpha=0.75)
    plt.axhline(OVR_FINAL_THRESHOLD, linestyle="--", alpha=0.5, label=f"threshold={OVR_FINAL_THRESHOLD}")
    plt.axhline(R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED, linestyle=":", alpha=0.5, label=f"weak r6g={R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED}")
    plt.xlabel("Spectrum index")
    plt.ylabel("Sigmoid probability")
    plt.title(f"{model_type.upper()} per-block target probabilities")
    plt.ylim(0, 1.05)
    plt.legend(loc="center right")
    plt.tight_layout()
    plt.show()

    # Plot average probabilities.
    plt.figure(figsize=(8, 5))
    bars = plt.bar(target_names, avg_probs * 100, alpha=0.8, edgecolor="black")
    plt.axhline(OVR_FINAL_THRESHOLD * 100, linestyle="--", alpha=0.5, label=f"threshold={OVR_FINAL_THRESHOLD*100:.0f}%")
    plt.axhline(R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED * 100, linestyle=":", alpha=0.5, label=f"weak r6g={R6G_WEAK_THRESHOLD_WHEN_MB_DETECTED*100:.0f}%")
    plt.ylabel("Average probability (%)")
    plt.title(f"{model_type.upper()} final target analyte probabilities")
    plt.ylim(0, 105)
    plt.legend()
    for bar, val in zip(bars, avg_probs):
        if val > 0.01:
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{val*100:.1f}%", ha="center", va="bottom")
    plt.tight_layout()
    plt.show()

    return avg_probs


# =========================================================
# MAIN
# =========================================================
def main():
    print("\n" + "=" * 70)
    print("Raman Transformer multi-label mixture detection")
    print("Same target-reporting logic as current PCA-SVM mixture branch")
    print("=" * 70)

    data_by_class, X_pure_raw, Y_pure_single = load_raw_training_data()
    X, Y = generate_synthetic_mixture_dataset(data_by_class, X_pure_raw, Y_pure_single)

    idx_train, idx_val, idx_test = random_split_indices(X.shape[0])
    print("\n" + "=" * 70)
    print("Step 3: Split")
    print("=" * 70)
    print(f"  train={len(idx_train)} val={len(idx_val)} test={len(idx_test)}")

    models_to_run = ["cnn", "transformer"] if MODEL_TYPE.lower() == "both" else [MODEL_TYPE.lower()]
    runtime_test_file, runtime_dark_file = _parse_cli_args()

    all_results = {}
    for mt in models_to_run:
        model, standardizer, model_dir, test_metrics = train_one_model(mt, X, Y, idx_train, idx_val, idx_test)
        avg_probs = predict_real_file(model, standardizer, mt, runtime_test_file, runtime_dark_file, debug=True)
        all_results[mt] = {
            "test_exact_match": test_metrics["exact_match"],
            "test_macro_f1": test_metrics["macro_f1"],
            "avg_probs": None if avg_probs is None else {target_names[i]: float(avg_probs[i]) for i in range(NUM_TARGETS)},
        }

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    for mt, res in all_results.items():
        print(f"  {mt.upper():12s} test_exact={res['test_exact_match']:.4f} test_macro_f1={res['test_macro_f1']:.4f}")
        if res["avg_probs"] is not None:
            print("    real-file avg probs:", res["avg_probs"])
    print("=" * 70)


if __name__ == "__main__":
    main()


DEVICE = cpu
MODEL_TYPE = transformer
OUTPUT_DIR = C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\_Raman_multilabel_CNN_Transformer_target_only

Raman Transformer multi-label mixture detection
Same target-reporting logic as current PCA-SVM mixture branch

Step 1: Load baseline_raw pure data
  r6g                  shape=(113, 1958)
  methylene_blue       shape=(118, 1958)
  adenine              shape=(110, 1958)
  di_water             shape=(115, 1958)

  Pure raw total: (456, 1958)
  Target analytes: ['r6g', 'methylene_blue', 'adenine']
  Background classes: ['di_water']

Step 2: Generate synthetic mixtures from baseline_raw

  Binary combinations: 6
    r6g + methylene_blue: 200
    r6g + adenine: 200
    r6g + di_water: 200
    methylene_blue + adenine: 200
    methylene_blue + di_water: 200
    adenine + di_water: 200

  Extra low-ratio r6g + methylene_blue mixtures:
    r6g=0.01, methylene_blue=0.99: 60
    r6g=0.02, methylene_blue=0.98

KeyboardInterrupt: 